# 06 - Model Evaluation and Persistence

**Objectif :** évaluer le modèle tuné sauvegardé sur le test final untouched, puis persister les rapports finaux.

In [ ]:
import pandas as pd
import plotly.express as px

from credit_risk_lab.config.settings import settings

print(f"Project root: {settings.project_root}")
print(f"Environment: {settings.environment}")

## 1. Load tuned model bundle

In [ ]:
from credit_risk_lab.infrastructure.modeling import JoblibModelBundleRepository

repository = JoblibModelBundleRepository()
bundle = repository.load(settings.model_bundle_path)
bundle["metadata"]

## 2. Load untouched external test

In [ ]:
from credit_risk_lab.infrastructure import CreditRiskQualityChecker
from credit_risk_lab.infrastructure.data_sources import CsvLoanDataLoader

raw_test_df = CsvLoanDataLoader(path=settings.raw_test_path).load()
clean_test_df = CreditRiskQualityChecker().clean(raw_test_df)

{
    "raw_test_path": str(settings.raw_test_path),
    "raw_test_rows": len(raw_test_df),
    "clean_test_rows": len(clean_test_df),
}

## 3. Score external test

In [ ]:
from credit_risk_lab.application import RawLoanScorer

scorer = RawLoanScorer(bundle)
scoring_result = scorer.score(clean_test_df)
y_test = clean_test_df[settings.target_column].astype(int)
test_probabilities = scoring_result.probabilities
sensitive_test = clean_test_df[
    [column for column in settings.sensitive_columns if column in clean_test_df]
]

{
    "model": scoring_result.model_name,
    "threshold": scoring_result.threshold,
    "scored_rows": len(test_probabilities),
}

## 4. Final test metrics

In [ ]:
from credit_risk_lab.infrastructure.evaluation import CreditRiskModelEvaluator

evaluator = CreditRiskModelEvaluator()
test_metrics = evaluator.metrics_frame(
    scoring_result.model_name,
    y_test,
    test_probabilities,
    scoring_result.threshold,
)
test_metrics.to_csv(settings.reports_dir / "external_test_metrics.csv", index=False)
test_metrics.round(4)

## 5. Confusion matrix and ROC-AUC

In [ ]:
from credit_risk_lab.infrastructure.visualization import plot_confusion_matrix_and_roc

plot_confusion_matrix_and_roc(
    y_test,
    test_probabilities,
    threshold=scoring_result.threshold,
    model_name=scoring_result.model_name,
).show()

## 6. Threshold trade-off analysis

In [ ]:
from credit_risk_lab.infrastructure.evaluation import (
    ThresholdAnalysisConfig,
    ThresholdAnalyzer,
)
from credit_risk_lab.infrastructure.visualization import plot_threshold_tradeoff

threshold_config = ThresholdAnalysisConfig(
    start=0.02,
    stop=0.42,
    step=0.04,
)
threshold_grid = ThresholdAnalyzer(threshold_config).grid(y_test, test_probabilities)
threshold_grid.to_csv(
    settings.reports_dir / "external_test_threshold_grid.csv",
    index=False,
)

print(threshold_grid.to_string(index=False))
plot_threshold_tradeoff(
    threshold_grid,
    selected_threshold=scoring_result.threshold,
).show()

## 7. Lift, gain, and accumulation analysis

In [ ]:
from credit_risk_lab.infrastructure.evaluation import lift_gain_table
from credit_risk_lab.infrastructure.visualization import plot_lift_gain_accumulation

lift_gain = lift_gain_table(y_test, test_probabilities, bins=10)
lift_gain.to_csv(settings.reports_dir / "external_test_lift_gain.csv", index=False)

display(lift_gain.round(4))
plot_lift_gain_accumulation(lift_gain).show()

## 8. Bootstrap confidence intervals

In [ ]:
from credit_risk_lab.infrastructure.evaluation import bootstrap_metric_intervals
from credit_risk_lab.infrastructure.visualization import plot_metric_confidence_intervals

metric_intervals = bootstrap_metric_intervals(
    y_test,
    test_probabilities,
    scoring_result.threshold,
    n_bootstrap=300,
    confidence_level=0.95,
    random_state=settings.random_state,
)
metric_intervals.to_csv(
    settings.reports_dir / "external_test_metric_intervals.csv",
    index=False,
)

display(metric_intervals.round(4))
plot_metric_confidence_intervals(metric_intervals).show()

## 9. Calibration

In [ ]:
from credit_risk_lab.infrastructure.evaluation import CalibrationEvaluator
from credit_risk_lab.infrastructure.visualization import plot_calibration

calibration = CalibrationEvaluator(bins=10).evaluate(
    y_test,
    test_probabilities,
)
calibration.to_csv(settings.reports_dir / "external_test_calibration.csv", index=False)

display(calibration.round(4))
plot_calibration(calibration, scoring_result.model_name).show()

## 10. Fairness diagnostics

In [ ]:
from credit_risk_lab.infrastructure.evaluation import FairnessEvaluator

fairness = FairnessEvaluator(min_group_size=30).evaluate(
    y_test,
    test_probabilities,
    sensitive_test,
    scoring_result.threshold,
)
fairness.to_csv(settings.reports_dir / "external_test_fairness.csv", index=False)
fairness

## 11. Persist final metadata report

In [ ]:
from credit_risk_lab.infrastructure.modeling import sha256_file

final_metadata = {
    **bundle["metadata"],
    "external_test_metrics": test_metrics.iloc[0].to_dict(),
    "external_test_dataset_sha256": sha256_file(settings.raw_test_path),
}

pd.Series(final_metadata, name="value").to_frame()